In [ ]:
import numpy as np

from factors import *

In [ ]:
bt = BackTester(fc_name_list=['fac_cumret', 'fac_upperline', 'fac_winrate'],
                instrument_type='futures_continuous_contract',
                instrument_id_list = ['C0', 'FG0'],
                fc_freq = '1d',
                start_time='20230101',
                end_time='20260310',
                portfolio_adjust_method = '1D',
                interest_method = 'compound',
                risk_free_rate=False,
                n_jobs = 5
                )
# bt.backtest()

In [ ]:
bt.data

In [ ]:
# self = bt
# Data = self.data
# df = Data.copy()
#
# df = df.sort_values(by='time', ascending=True)
# df = df.set_index(['time', 'instrument_id'])
#
# fc_class_list = [resolve_factor_class(fc_name) for fc_name in self.fc_name_list]
# f = lambda x: get_factor_value_for_one_factor(df, x)
#
# with Parallel(n_jobs=self.n_jobs) as parallel:
#     mapper_list = parallel(delayed(f)(fc_class) for fc_class in fc_class_list)
# mapper_list = [x.reset_index() for x in mapper_list]

In [ ]:
bt.fc_name_with_param_list

In [ ]:
df = bt.data.copy()

In [ ]:
bt.performance_dc['C0']['fac_cumret_1_50']['daily_gross_ret'].copy()

In [ ]:
df[fc_col].ffill().fillna(0)

In [ ]:
bt.ts_performance_summary

In [ ]:
bt.plot_nav()

In [ ]:
from tsfresh.examples.robot_execution_failures import download_robot_execution_failures, \
    load_robot_execution_failures
download_robot_execution_failures()
timeseries, y = load_robot_execution_failures()

In [ ]:
print(timeseries.head())

In [ ]:
from tsfresh import extract_features
extracted_features = extract_features(timeseries, column_id="id", column_sort="time")

In [ ]:
timeseries

In [ ]:
extracted_features

In [ ]:
extracted_features.columns

In [ ]:
from factors.factor_auto_search import FactorGenerator

In [ ]:
from factors.factor_auto_search import FactorGenerator

fg = FactorGenerator(
    method='tsfresh',
    instrument_id_list=['C0', 'FG0'],
    fc_freq='1d',
    start_time='20230101',
    end_time='20260310',
    min_window_size=20,
    max_factor_count=200,
    tsfresh_profile='comprehensive',
    n_jobs=5,
    apply_rolling_norm=True,
    rolling_norm_window=30,
    rolling_norm_min_periods=20
)



In [ ]:
generated_df = fg.generate()
fc_subset = fg.generated_fc_name_list
print(len(fg.generated_fc_name_list))
fg.save_fc_value(fc_subset, file_name='tsfresh_fc_subset', file_format='parquet')
bt = fg.backtest(fc_name_list=fc_subset)

In [ ]:
len(fg.generated_fc_name_list)

In [ ]:
bt.plot_nav()

In [ ]:
fg.generated_fc_name_list

In [ ]:
fc_subset = ['close__maximum', 'close__absolute_maximum',
             'position__standard_deviation', 'position__variance']
config_path = fg.save_fc(fc_subset)
selected_fc = FactorGenerator.load_fc(config_path)
print(selected_fc)

In [ ]:
bt2 = fg.backtest_from_fc_config(config_path)

In [ ]:
bt2.performance_summary.loc[bt2.performance_summary['Instrument ID'] == 'C0']

In [ ]:
bt2.plot_nav()

# auto search

## 开始search

In [ ]:
from factors.factor_auto_search import FactorGenerator
fg = FactorGenerator(
    method='tsfresh',
    instrument_id_list=['C0'],
    fc_freq='1d',
    start_time='20230101',
    end_time='20260310',
    min_window_size=20,
    max_factor_count=20000,
    tsfresh_profile='comprehensive',
    n_jobs=5,
    apply_rolling_norm=True,
    rolling_norm_window=30,
    rolling_norm_min_periods=20
)
# one-step: mine + filter + save high-quality config
result = fg.auto_mine_select_and_save_fc(
    net_ret_threshold=0.05,
    sharpe_threshold=0.8,
    fc_package_name='tsfresh_high_quality_fc_20260317_new',
    require_all_instruments=False
)

## 检查信息泄露

In [ ]:
# 这个代码跑了一个半小时才跑出结果
leakage_check = fg.check_if_leakage(fc_name_list=result['selected_fc_name_list'], raise_error=True)

In [ ]:
leakage_check

In [ ]:
result.keys()

In [ ]:
result['selected_fc_name_list']

In [ ]:
fg.generated_data['close__mean_second_derivative_central']

In [ ]:
df_ps = result['bt'].performance_summary
df_ps.loc[(df_ps['Instrument ID'] == 'C0') & (df_ps['Factor Name'].isin(result['selected_fc_name_list'])]

In [ ]:
result['bt'].plot_nav(fc_name=result['selected_fc_name_list'])

# tsfresh自动生成因子

In [ ]:
from factors.factor_auto_search import FactorGenerator
fg1 = FactorGenerator(
    method='tsfresh',
    instrument_id_list=['C0'],
    fc_freq='1d',
    start_time='20230101',
    end_time='20260310',
    max_factor_count=20000,
    min_window_size=20,
    model_name='deepseek',
    llm_factor_count=5,
    llm_temperature=0.7,
    n_jobs=5,
    apply_rolling_norm=True,
    rolling_norm_window=30,
    rolling_norm_min_periods=20,
    calculate_baseline=True,
    version='test_20260317'
)

In [ ]:
# one-step: mine + filter + save high-quality config
result1 = fg1.auto_mine_select_and_save_fc(
    net_ret_threshold=0.05,
    sharpe_threshold=0.8,
    require_all_instruments=False
)

In [ ]:
print("全部生成的因子: ", fg1.generated_fc_name_list)
print("选中的因子: ", result1['selected_fc_name_list'])
print("计算出的因子值: ")
display(fg1.generated_data.iloc[100:110])
print("选中因子的回测结果: ")
display(result1['bt'].performance_summary)
print("选中因子的净值曲线: ")
result1['bt'].plot_nav(fc_name=result1['selected_fc_name_list'],
                                                 show_baseline=True)

## llm自动生成因子

In [ ]:
from factors.factor_auto_search import FactorGenerator
fg2 = FactorGenerator(
    method='llm_prompt',
    instrument_id_list=['C0'],
    fc_freq='1d',
    start_time='20200101',
    end_time='20241231',
    max_factor_count=200,
    min_window_size=20,
    model_name='deepseek',
    llm_factor_count=5,
    llm_temperature=0.7,
    n_jobs=5,
    apply_rolling_norm=True,
    rolling_norm_window=30,
    rolling_norm_min_periods=20,
    calculate_baseline=True,
    llm_early_stopping_round=20,  # 20次调用llm都无法产生有效因子，那么早停
    llm_user_requirement='多重视position和volume相关的逻辑，当然也可以结合一些价格数据。因为根据经验，position相关逻辑的因子效果比较好。因子逻辑可以适当复杂一些，因为简单的逻辑已经没有什么有用的因子了。',
    version='test_20260317'
)

In [ ]:
# one-step: mine + filter + save high-quality config
result2 = fg2.auto_mine_select_and_save_fc(
    net_ret_threshold=0.03,
    sharpe_threshold=0.5,
    require_all_instruments=False
)


In [ ]:
print("全部生成的因子: ", fg2.generated_fc_name_list)
print("选中的因子: ", result2['selected_fc_name_list'])
print("计算出的因子值: ")
display(fg2.generated_data.iloc[100:110])
print("选中因子的回测结果: ")
display(result2['bt'].performance_summary)
print("选中因子的净值曲线: ")
result2['bt'].plot_nav(fc_name=result2['selected_fc_name_list'],
                       show_baseline=True)

In [ ]:
result2['bt']

In [ ]:
result2['bt'].plot_nav(fc_name=fg2.generated_fc_name_list[:10],
                       show_baseline=True)

# 因子融合

In [ ]:
from factors.factor_auto_search import FactorFusioner
ff = FactorFusioner(version_list='test_20260317',
                    fused_fc_name_suffix='test_20260317',
                    fusion_strategy='average_weight',
                    instrument_type='futures_continuous_contract',
                    instrument_id_list='C0',
                    fc_freq='1d',
                    start_time='20230101',
                    end_time='20260310',
                    apply_rolling_norm=True,
                    apply_rolling_norm_after_fusion=False,  # 融合后一般不再做rolling norm了
                    calculate_baseline=True)
ff.generate()

In [ ]:
ff.raw_fc_value

In [ ]:
bt = ff.backtest()

In [ ]:
bt.ts_performance_summary

In [ ]:
bt.plot_nav(show_baseline=True)